# HIV Hi — Model complexity analysis

This notebook analyses whether the internal validation protocol used for hyperparameter selection changes the complexity of the selected models.

The comparison is between:

- OOD holdout inner validation
- Random shuffle inner validation

The main question is:

**Does random shuffle select models that are more complex and less calibrated for final OOD test generalization?**

The analysis uses the saved per-fold artifacts:

- `params_fold_i.json`
- `complexity_fold_i.json`

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("../..").resolve()

RAW_RESULTS_DIR = PROJECT_ROOT / "results" / "hi" / "hiv"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / "hi"
    / "hiv"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw results:", RAW_RESULTS_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: /home/f.capria/drug-discovery-lohi
Raw results: /home/f.capria/drug-discovery-lohi/results/hi/hiv
Output dir: /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/hiv


## Experiment registry

The registry defines which result folders correspond to each combination of:

- model
- fingerprint
- protocol

In [3]:
EXPERIMENTS = [
    # Decision Tree
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_hiv_hi_random_shuffle_maccs",
    },

    # Logistic Regression
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_rdkit_desc",
    },

    # Linear SVM
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_hiv_hi_random_shuffle_maccs",
    },
]

In [4]:
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def safe_get(dct, key, default=np.nan):
    return dct.get(key, default)

In [5]:
def load_complexity_rows(experiments, raw_results_dir):
    rows = []

    for exp in experiments:
        result_dir = raw_results_dir / exp["result_dir"]

        if not result_dir.exists():
            print(f"Missing directory: {result_dir}")
            continue

        for fold in [1, 2, 3]:
            params_path = result_dir / f"params_fold_{fold}.json"
            complexity_path = result_dir / f"complexity_fold_{fold}.json"

            if not params_path.exists():
                print(f"Missing params file: {params_path}")
                continue

            if not complexity_path.exists():
                print(f"Missing complexity file: {complexity_path}")
                continue

            params = read_json(params_path)
            complexity = read_json(complexity_path)

            train_metrics = params.get("train_metrics", {})
            test_metrics = params.get("test_metrics", {})

            inner_pr_auc = params.get("inner_selection_score", np.nan)
            inner_train_pr_auc = params.get("inner_train_score", np.nan)
            train_pr_auc = train_metrics.get("pr_auc", np.nan)
            test_pr_auc = test_metrics.get("pr_auc", np.nan)

            row = {
                "model": exp["model"],
                "model_short": exp["model_short"],
                "fingerprint": exp["fingerprint"],
                "protocol": exp["protocol"],
                "result_dir": exp["result_dir"],
                "fold": fold,

                # Scores
                "inner_pr_auc": inner_pr_auc,
                "inner_train_pr_auc": inner_train_pr_auc,
                "train_pr_auc": train_pr_auc,
                "test_pr_auc": test_pr_auc,

                # Gaps
                "train_inner_gap": train_pr_auc - inner_pr_auc,
                "inner_test_gap": inner_pr_auc - test_pr_auc,
                "train_test_gap": train_pr_auc - test_pr_auc,
            }

            # Add all complexity fields
            for key, value in complexity.items():
                row[key] = value

            rows.append(row)

    return pd.DataFrame(rows)

In [6]:
complexity_all = load_complexity_rows(EXPERIMENTS, RAW_RESULTS_DIR)

complexity_all = complexity_all.sort_values(
    ["model_short", "fingerprint", "protocol", "fold"]
).reset_index(drop=True)

complexity_all.head(3)

,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,...,l1_norm,l2_norm,approx_margin,intercept,C,penalty,l1_ratio,dual,n_support_per_class,n_support_total
0,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,1,0.112536,0.032367,0.1173,0.0395,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,2,0.087381,0.097746,0.2730,0.0748,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,3,0.083047,0.545721,0.5066,0.0257,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("Rows loaded:", len(complexity_all))
print("Expected rows:", len(EXPERIMENTS) * 3)

complexity_all[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
        "model_class",
    ]
]

Rows loaded: 42
Expected rows: 42


,model,fingerprint,protocol,fold,inner_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,model_class
0,Decision Tree,ECFP4,OOD holdout,1,0.112536,0.1173,0.0395,0.073036,0.0778,DecisionTreeClassifier
1,Decision Tree,ECFP4,OOD holdout,2,0.087381,0.2730,0.0748,0.012581,0.1982,DecisionTreeClassifier
2,Decision Tree,ECFP4,OOD holdout,3,0.083047,0.5066,0.0257,0.057347,0.4809,DecisionTreeClassifier
3,Decision Tree,ECFP4,Random shuffle,1,0.349622,0.4334,0.0893,0.260322,0.3441,DecisionTreeClassifier
4,Decision Tree,ECFP4,Random shuffle,2,0.359180,0.3868,0.0616,0.297580,0.3252,DecisionTreeClassifier
5,Decision Tree,ECFP4,Random shuffle,3,0.393590,0.5079,0.0279,0.365690,0.4800,DecisionTreeClassifier
6,Decision Tree,MACCS,OOD holdout,1,0.121836,0.2458,0.1470,-0.025164,0.0988,DecisionTreeClassifier
7,Decision Tree,MACCS,OOD holdout,2,0.208302,0.3699,0.0613,0.147002,0.3086,DecisionTreeClassifier
8,Decision Tree,MACCS,OOD holdout,3,0.208163,0.4180,0.0250,0.183163,0.3930,DecisionTreeClassifier
9,Decision Tree,MACCS,Random shuffle,1,0.332829,0.4744,0.0584,0.274429,0.4160,DecisionTreeClassifier


## Logistic Regression complexity table

For Logistic Regression complexity is described by:

- selected `C` - larger --> weaker regularization
- selected `l1_ratio`
- number of non-zero coefficients
- sparsity
- L1 norm of the coefficient vector
- L2 norm of the coefficient vector - 

In [8]:
lr_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "l1_ratio",
    "class_weight",
    "n_nonzero_coefficients",
    "sparsity",
    "l1_norm",
    "l2_norm",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

lr_table = (
    complexity_all[complexity_all["model_short"] == "LR"]
    [lr_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

lr_table

,model,fingerprint,protocol,fold,C,l1_ratio,class_weight,n_nonzero_coefficients,sparsity,l1_norm,l2_norm,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Logistic Regression,ECFP4,OOD holdout,1,0.10,1.0,NaN,105.0,0.897461,22.392329,3.301251,0.102488,0.0676,0.034888,0.3376
1,Logistic Regression,ECFP4,OOD holdout,2,0.10,0.0,balanced,1024.0,0.000000,397.548640,15.379072,0.117646,0.0752,0.042446,0.3511
2,Logistic Regression,ECFP4,OOD holdout,3,0.10,1.0,balanced,974.0,0.048828,167.780383,8.996314,0.165258,0.0302,0.135058,0.4986
3,Logistic Regression,ECFP4,Random shuffle,1,0.10,0.0,NaN,1024.0,0.000000,150.262848,6.016560,0.450767,0.0906,0.360167,0.5100
4,Logistic Regression,ECFP4,Random shuffle,2,0.10,0.0,NaN,1024.0,0.000000,136.423495,5.451001,0.448193,0.0954,0.352793,0.4445
5,Logistic Regression,ECFP4,Random shuffle,3,0.10,0.0,NaN,1024.0,0.000000,158.155419,6.300322,0.526874,0.0281,0.498774,0.6306
6,Logistic Regression,MACCS,OOD holdout,1,10.00,1.0,balanced,164.0,0.017964,187.597706,23.835826,0.121724,0.1984,-0.076676,-0.0205
7,Logistic Regression,MACCS,OOD holdout,2,0.10,1.0,NaN,52.0,0.688623,9.456506,1.878584,0.158481,0.0929,0.065581,0.2168
8,Logistic Regression,MACCS,OOD holdout,3,0.10,1.0,balanced,157.0,0.059880,57.068311,6.778976,0.245719,0.0473,0.198419,0.1495
9,Logistic Regression,MACCS,Random shuffle,1,5.00,0.5,NaN,162.0,0.029940,74.750326,8.145833,0.304843,0.1634,0.141443,0.1841


## Linear SVM complexity table

For linear SVM the indicators are:

- selected `C`
- L2 norm of the weight vector
- approximate margin

In [9]:
svm_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "class_weight",
    "l2_norm",
    "approx_margin",
    "n_nonzero_coefficients",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

svm_table = (
    complexity_all[complexity_all["model_short"] == "SVM"]
    [svm_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

svm_table

,model,fingerprint,protocol,fold,C,class_weight,l2_norm,approx_margin,n_nonzero_coefficients,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,SVM linear,ECFP4,OOD holdout,1,0.10,NaN,3.612329,0.276830,1024.0,0.077855,0.0616,0.016255,0.5632
1,SVM linear,ECFP4,OOD holdout,2,0.50,NaN,7.155549,0.139752,1024.0,0.111076,0.0675,0.043576,0.5762
2,SVM linear,ECFP4,OOD holdout,3,0.01,balanced,4.482761,0.223077,1024.0,0.105710,0.0306,0.075110,0.5123
3,SVM linear,ECFP4,Random shuffle,1,0.01,balanced,5.114753,0.195513,1024.0,0.397047,0.1223,0.274747,0.3461
4,SVM linear,ECFP4,Random shuffle,2,0.10,NaN,2.918730,0.342615,1024.0,0.395126,0.0721,0.323026,0.5026
5,SVM linear,ECFP4,Random shuffle,3,0.10,NaN,4.144292,0.241296,1024.0,0.507792,0.0241,0.483692,0.6523
6,SVM linear,MACCS,OOD holdout,1,0.10,balanced,5.167922,0.193501,164.0,0.099913,0.1456,-0.045687,0.0555
7,SVM linear,MACCS,OOD holdout,2,0.10,NaN,0.007764,128.799946,163.0,0.214118,0.1210,0.093118,0.1425
8,SVM linear,MACCS,OOD holdout,3,0.01,balanced,2.681085,0.372983,163.0,0.174833,0.0437,0.131133,0.2296
9,SVM linear,MACCS,Random shuffle,1,0.01,NaN,0.004461,224.142335,164.0,0.215990,0.1738,0.042190,0.0548


## Decision Tree complexity table

For Decision Trees:

- selected `ccp_alpha`
- selected `max_depth`
- effective tree depth
- number of nodes
- number of leaves
- number of features used in the tree
- average minimum depth of the used features

The number of nodes and leaves gives a direct indication of how large the fitted tree is.

In [10]:
dt_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "ccp_alpha",
    "max_depth",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "used_feature_fraction",
    "feature_min_depth_mean",
    "feature_min_depth_std",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

dt_table = (
    complexity_all[complexity_all["model_short"] == "DT"]
    [dt_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

dt_table

,model,fingerprint,protocol,fold,ccp_alpha,max_depth,effective_depth,n_nodes,n_leaves,n_features_used,used_feature_fraction,feature_min_depth_mean,feature_min_depth_std,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.0000,3.0,3.0,15.0,8.0,7.0,0.006836,1.428571,0.728431,0.112536,0.0395,0.073036,0.0778
1,Decision Tree,ECFP4,OOD holdout,2,0.0000,15.0,15.0,351.0,176.0,145.0,0.141602,8.806897,3.574424,0.087381,0.0748,0.012581,0.1982
2,Decision Tree,ECFP4,OOD holdout,3,0.0001,15.0,15.0,419.0,210.0,174.0,0.169922,8.413793,3.410773,0.083047,0.0257,0.057347,0.4809
3,Decision Tree,ECFP4,Random shuffle,1,0.0000,15.0,15.0,475.0,238.0,181.0,0.176758,8.309392,3.235400,0.349622,0.0893,0.260322,0.3441
4,Decision Tree,ECFP4,Random shuffle,2,0.0000,15.0,15.0,355.0,178.0,143.0,0.139648,8.174825,3.362306,0.359180,0.0616,0.297580,0.3252
5,Decision Tree,ECFP4,Random shuffle,3,0.0000,15.0,15.0,611.0,306.0,230.0,0.224609,8.673913,3.157148,0.393590,0.0279,0.365690,0.4800
6,Decision Tree,MACCS,OOD holdout,1,0.0001,8.0,8.0,173.0,87.0,70.0,0.419162,4.885714,1.669413,0.121836,0.1470,-0.025164,0.0988
7,Decision Tree,MACCS,OOD holdout,2,0.0000,15.0,15.0,537.0,269.0,114.0,0.682635,7.508772,3.255887,0.208302,0.0613,0.147002,0.3086
8,Decision Tree,MACCS,OOD holdout,3,0.0000,10.0,10.0,413.0,207.0,110.0,0.658683,6.054545,2.057520,0.208163,0.0250,0.183163,0.3930
9,Decision Tree,MACCS,Random shuffle,1,0.0000,15.0,15.0,863.0,432.0,136.0,0.814371,7.110294,2.772429,0.332829,0.0584,0.274429,0.4160


## Gap analysis table

This table collects the three main performance levels:

- train PR-AUC
- inner validation PR-AUC
- final OOD test PR-AUC

and the corresponding gaps:

$$\text{train-inner gap} = \text{train PR-AUC} - \text{inner PR-AUC}$$

$$\text{inner-test gap} = \text{inner PR-AUC} - \text{test PR-AUC}$$

$$\text{train-test gap} = \text{train PR-AUC} - \text{test PR-AUC}$$


This table connects model selection, overfitting and OOD generalization.

In [11]:
gap_cols = [
    "model",
    "model_short",
    "fingerprint",
    "protocol",
    "fold",
    "train_pr_auc",
    "inner_pr_auc",
    "inner_train_pr_auc",
    "test_pr_auc",
    "train_inner_gap",
    "inner_test_gap",
    "train_test_gap",
]

gap_analysis = (
    complexity_all[gap_cols]
    .sort_values(["model_short", "fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

gap_analysis

,model,model_short,fingerprint,protocol,fold,train_pr_auc,inner_pr_auc,inner_train_pr_auc,test_pr_auc,train_inner_gap,inner_test_gap,train_test_gap
0,Decision Tree,DT,ECFP4,OOD holdout,1,0.1173,0.112536,0.032367,0.0395,0.004764,0.073036,0.0778
1,Decision Tree,DT,ECFP4,OOD holdout,2,0.2730,0.087381,0.097746,0.0748,0.185619,0.012581,0.1982
2,Decision Tree,DT,ECFP4,OOD holdout,3,0.5066,0.083047,0.545721,0.0257,0.423553,0.057347,0.4809
3,Decision Tree,DT,ECFP4,Random shuffle,1,0.4334,0.349622,0.407691,0.0893,0.083778,0.260322,0.3441
4,Decision Tree,DT,ECFP4,Random shuffle,2,0.3868,0.359180,0.410228,0.0616,0.027620,0.297580,0.3252
5,Decision Tree,DT,ECFP4,Random shuffle,3,0.5079,0.393590,0.513710,0.0279,0.114310,0.365690,0.4800
6,Decision Tree,DT,MACCS,OOD holdout,1,0.2458,0.121836,0.093290,0.1470,0.123964,-0.025164,0.0988
7,Decision Tree,DT,MACCS,OOD holdout,2,0.3699,0.208302,0.163001,0.0613,0.161598,0.147002,0.3086
8,Decision Tree,DT,MACCS,OOD holdout,3,0.4180,0.208163,0.418451,0.0250,0.209837,0.183163,0.3930
9,Decision Tree,DT,MACCS,Random shuffle,1,0.4744,0.332829,0.499108,0.0584,0.141571,0.274429,0.4160


## Aggregated complexity summary


In [12]:
summary_metrics = [
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
    "l2_norm",
    "n_nonzero_coefficients",
    "approx_margin",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "feature_min_depth_mean",
]

available_summary_metrics = [
    col for col in summary_metrics
    if col in complexity_all.columns
]

complexity_summary = (
    complexity_all
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    [available_summary_metrics]
    .agg(["mean", "std"])
)

complexity_summary.columns = [
    "_".join([c for c in col if c])
    for col in complexity_summary.columns
]

complexity_summary = complexity_summary.reset_index()

complexity_summary

,index,model,model_short,fingerprint,protocol,inner_pr_auc_mean,inner_pr_auc_std,test_pr_auc_mean,test_pr_auc_std,inner_test_gap_mean,...,effective_depth_mean,effective_depth_std,n_nodes_mean,n_nodes_std,n_leaves_mean,n_leaves_std,n_features_used_mean,n_features_used_std,feature_min_depth_mean_mean,feature_min_depth_mean_std
0,0,Decision Tree,DT,ECFP4,OOD holdout,0.094321,0.015922,0.046667,0.025322,0.047655,...,11.0,6.928203,261.666667,216.308422,131.333333,108.154211,108.666667,89.231908,6.216420,4.151055
1,1,Decision Tree,DT,ECFP4,Random shuffle,0.367464,0.023125,0.059600,0.030749,0.307864,...,15.0,0.000000,480.333333,128.083306,240.666667,64.041653,184.666667,43.615746,8.386043,0.258222
2,2,Decision Tree,DT,MACCS,OOD holdout,0.179434,0.049881,0.077767,0.062645,0.101667,...,11.0,3.605551,374.333333,185.054947,187.666667,92.527473,98.000000,24.331050,6.149677,1.314114
3,3,Decision Tree,DT,MACCS,Random shuffle,0.389259,0.049023,0.053633,0.014836,0.335626,...,15.0,0.000000,869.666667,90.184995,435.333333,45.092498,130.333333,5.131601,6.936482,0.261569
4,4,Logistic Regression,LR,ECFP4,OOD holdout,0.128464,0.032753,0.057667,0.024088,0.070797,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,Logistic Regression,LR,ECFP4,Random shuffle,0.475278,0.044702,0.071367,0.037547,0.403911,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,Logistic Regression,LR,MACCS,OOD holdout,0.175308,0.063687,0.112867,0.077504,0.062442,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,Logistic Regression,LR,MACCS,Random shuffle,0.342359,0.042536,0.100133,0.059422,0.242225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,Logistic Regression,LR,RDKit desc,OOD holdout,0.172062,0.047645,0.088500,0.047301,0.083562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,Logistic Regression,LR,RDKit desc,Random shuffle,0.377856,0.066132,0.088733,0.045302,0.289123,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
complexity_all.to_csv(OUTPUT_DIR / "complexity_all.csv", index=False)
lr_table.to_csv(OUTPUT_DIR / "complexity_lr.csv", index=False)
svm_table.to_csv(OUTPUT_DIR / "complexity_svm.csv", index=False)
dt_table.to_csv(OUTPUT_DIR / "complexity_dt.csv", index=False)
gap_analysis.to_csv(OUTPUT_DIR / "complexity_gap_analysis.csv", index=False)
complexity_summary.to_csv(OUTPUT_DIR / "complexity_summary.csv", index=False)

print("Saved complexity tables in:")
print(OUTPUT_DIR)

Saved complexity tables in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/hiv
